# Pose predictors

## File Handling
To run predictions a `Scanned3DEnvironment` object and a `HeadsetRecording` object or a single frame is needed, those can be loaded from folders or created.

### Creation of RobotEnvironment and HeadsetData
Those 2 datatypes can be created from an GatheredRobotData object and a .vrs file respectively.

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)

from shared import CompleteRobotScan
from headset_localization import (
    HeadsetRecording, bind_headset_recording_to_scan, Scanned3dEnvironment, XYZImageGenerationConfig, ICPAlignmentConfig, visualize_robot_camera_environment_combo
)

In [ ]:
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"

robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
robot_data_from_disk = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=True)
)

labeled_headset_data = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

visualize_loaded_data = False

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_data_from_disk, headset_data=labeled_headset_data)

### Saving and loading RobotEnvironments and HeadsetData
`Scanned3dEnvironment` and `HeadsetRecording` both have file interfaces which can be used to load/save them from/to the disk

In [ ]:
processed_datasets_location = "../processed_datasets"

labeled_headset_data.save(processed_datasets_location, new_name = "quickstart_headset_data")
robot_data_from_disk.save(processed_datasets_location, new_name = "quickstart_robot_data")

robot_env_from_disk = Scanned3dEnvironment.from_folder(f"{processed_datasets_location}/quickstart_robot_data")
headset_data_from_disk = HeadsetRecording.from_folder(f"{processed_datasets_location}/quickstart_headset_data")

### Alternative: Creating from TU-München Dataset
Alternative they cam be created from a TU-München Dataset: https://cvg.cit.tum.de/data/datasets/rgbd-dataset

In [ ]:
from headset_localization import scanned_3d_environment_and_headset_recording_from_tum

tum_rgbd_dataset_location = "../tum_datasets/rgbd_dataset_freiburg2_desk"

tum_robot_env, tum_headset_data = scanned_3d_environment_and_headset_recording_from_tum(
        folder=tum_rgbd_dataset_location,
        rgb_camera_name="freiburg2",
        time_tolerance= 0.03,
        n_robot_images= 20,
        xyz_image_generation_config=XYZImageGenerationConfig(),
        xyz_image_alginment_config=ICPAlignmentConfig(do_alginment=False),
        intervall=(0.0, 0.1)
)

### Visualising Datasets
Robot environments and Headset datasets can be visualized in 3d

In [ ]:
vis_robot_env, vis_headset, vis_both, vis_tum = False, False, True, True

if vis_robot_env:
    robot_env_from_disk.visualize_3d_data()
if vis_headset:
    headset_data_from_disk.visualize_3d_data()
if vis_both:
    visualize_robot_camera_environment_combo(robot_env=robot_env_from_disk, headset_data=headset_data_from_disk)

if vis_tum:
    visualize_robot_camera_environment_combo(robot_env=tum_robot_env, headset_data=tum_headset_data, headset_frame_size=0.05)

## Testing Predictors

In [ ]:
from headset_localization import (
    PnPLocalizer, ExtractAndMatchWrapperConfig, ExtractAndLightGlue,
    PredictionOnDataset, GradableLocalizer, PnPLLocalizer, EllipsoidLocalizer, NPredictors1DatasetGrader
)

### Creating Localizers
Now an `HeadsetLocalizer` can be created. An `HeadsetLocalizer` instance is build upon an `Scanned3DEnvironment` instance and can predict positions from headset-images.

In [ ]:
chosen_headset_data = headset_data_from_disk
chosen_robot_env = robot_env_from_disk

# Simple Point only predictor
point_predictor = PnPLocalizer(
        cam2_intrinsic_mtx=chosen_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=chosen_robot_env.robot_bgr_images,
        cam1_xyz_images=chosen_robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(extract_and_match=ExtractAndLightGlue())
)

# Predicting a single frame:
base_t_headset = point_predictor.est_base_t_cam2(
    cam2_bgr_image=chosen_headset_data.bgr_image_s[0],
)

### Grading the Performance of an Initialised Predictor:

In [ ]:
init_predictor_grade = PredictionOnDataset(
    predictor = point_predictor,
    headset_data = chosen_headset_data,
    number_retry = 1
)

init_predictor_grade.print_summary()

The predictions can also be visualized in 3d

In [ ]:
visualize_prediction = True
if visualize_prediction:
    init_predictor_grade.visualize_predictions(
        robot_env=chosen_robot_env,
        show_label=False
    )

### Grading of multiple uninitialised Localizers in an Environment
Localizers provide `get_creation_function` methods, which can be used to initialize one, to grade the creation behaviour and time. Their signatures are similar to the of `__init__`. `GradableLocalizer` provided additional configuration options on how many retries per prediction and the names / identifiers for plotting.

In [ ]:
from headset_localization import *

# Creation of the Predictors
points_light_glue = GradableLocalizer(
    creator= PnPLocalizer.get_creation_function(
        cam2_intrinsic_mtx=chosen_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.4]
        )
    ),
    name="PnP-LG"
)

points_loma = GradableLocalizer(
    creator= PnPLocalizer.get_creation_function(
        cam2_intrinsic_mtx=chosen_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndMatchLoMa('LoMaB128'),
            crop_augmentations=[0.2]
        )
    ),
    name="PnP-LoMa"
)

points_lines_light_glue = GradableLocalizer(
    creator=PnPLLocalizer.get_creation_function(
        cam2_intrinsic_mtx = chosen_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.4]
        ),
        cam1_line_generator = LineGenerator(lsd_diagonal_size=850),
    ),
    name="PnP+L-LG"
)

points_lines_loma = GradableLocalizer(
    creator=PnPLLocalizer.get_creation_function(
        cam2_intrinsic_mtx = chosen_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config = ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndMatchLoMa('LoMaB128'),
            crop_augmentations=[0.4]
        ),
        cam1_line_generator = LineGenerator(lsd_diagonal_size=850),
    ),
    name="PnP+L-LoMa"
)

ellipsoids_light_glue = GradableLocalizer(
    creator=EllipsoidLocalizer.get_creation_function(
        cam2_intrinsic_mtx = chosen_headset_data.intrinsic_cam_mtx,
        matching_config=GaussianMatchingConfig(dummy_value=0.001),
        cam1_segmenter = SAM3Segmenter(Sam3Prompt(mask_threshold=0.2)),
        ellipsoid_fitter = MVEEEllipsoidFitter(contamination=0.2),
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.4],
        ),
        ellipsoid_matching_config=PointCloudMatchingConfig(min_cluster_size=2),
    ),
    name="Ellipsoids"
)

Now those can be used to create a grader object for multiple `HeadsetLocalizer` variants.

In [ ]:
grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=[points_light_glue, points_loma, ellipsoids_light_glue, points_lines_light_glue, points_lines_loma],
    headset_data = chosen_headset_data,
    robot_env = chosen_robot_env,
    compute_ray_intersection_error=True, use_tqdm_for_frames=False, use_tqdm_for_predictors=True
)


In [ ]:
from headset_localization import SingleValueErrorType

if True:
    grader.visualize_predictions_3d()

grader.print_summary()

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,1, figsize = (12,6))
grader.plot_creation_times(ax)

fig1, ax = plt.subplots(1, 1, figsize = (4, 3))
grader.plot_error_vs_error(ax, SingleValueErrorType.AVG_TRANSLATIONAL, SingleValueErrorType.AVG_ROTATIONAL, custom_title = "Avg. translational vs. rotational error", plot_legend=True, adjust_texts = True, use_in_plot_text=False)
fig1, ax = plt.subplots(1, 1, figsize = (4, 3))
grader.plot_error_vs_error(ax, SingleValueErrorType.MED_TRANSLATIONAL, SingleValueErrorType.MED_ROTATIONAL, custom_title = "Med. translational vs. rotational error", plot_legend=False, adjust_texts = True, use_in_plot_text=False)

fig1, ax = plt.subplots(1, 1, figsize = (15, 3.5))
grader.plot_prediction_times(ax=ax, rotate_x_labels=None)

grader.print_error_under_limits(limits_m=[0.05, 0.1, 0.5], error_type='ATE')
grader.print_error_under_limits(limits_m=[0.05, 0.1, 0.5], error_type='ARE')

fig1, axes = plt.subplots(2, 3, figsize = (15, 8))

grader.plot_signed_error_comparison(axes = [axes[0,0], axes[0,1], axes[0,2], axes[1,0], axes[1,1], axes[1,2]],explain = True)

### Other Datasets

In [ ]:
scans = [CompleteRobotScan.from_folder(scan_loc) for scan_loc in ["../example_datasets/example_small_aruco1", "../example_datasets/small_aruco_2"]]
envs = [
        Scanned3dEnvironment.from_gathered_robot_data(
                robot_data = scan,
                number_of_sampled_datapoints=20,
                sample_datapoints_based_on_aruco_corectness = False,
                only_sample_robot_datapoints_w_marker_estimates = True,
                markers_use_advanced_removal=True,
                est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
                est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=True)
        )
    for scan in scans
]

scene_recording_pairs = [
    (env, bind_headset_recording_to_scan(headset_data = HeadsetRecording.from_vrs_file(rec_loc), robot_data = scan))
    for env, scan, rec_loc in zip(
        [envs[0], envs[0], envs[1], envs[1]],
        [scans[0], scans[0], scans[1], scans[0]], 
        [
            "../example_datasets/small_aruco1_sitting_20fps.vrs",
            "../example_datasets/small_aruco1_standing_20fps.vrs",
            "../example_datasets/small_aruco_2_1.vrs",
            "../example_datasets/small_aruco_2_2.vrs"
        ]
    )
]

In [ ]:
for name, (scanned_3d_env, headset_recording) in zip(["aruco1-sitting", "aruco1-standing", "aruco2-1", "aruco2-2"], scene_recording_pairs):
    points_light_glue = GradableLocalizer(
        creator= PnPLocalizer.get_creation_function(
            cam2_intrinsic_mtx=headset_recording.intrinsic_cam_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(extract_and_match=ExtractAndLightGlue(), crop_augmentations=[0.4])
        ),
        name="PnP-LG"
    )

    points_loma = GradableLocalizer(
        creator= PnPLocalizer.get_creation_function(
            cam2_intrinsic_mtx=headset_recording.intrinsic_cam_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(extract_and_match=ExtractAndMatchLoMa('LoMaB128'), crop_augmentations=[0.2])
        ),
        name="PnP-LoMa"
    )

    points_lines_light_glue = GradableLocalizer(
        creator=PnPLLocalizer.get_creation_function(
            cam2_intrinsic_mtx = headset_recording.intrinsic_cam_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(extract_and_match=ExtractAndLightGlue(), crop_augmentations=[0.4]),
            cam1_line_generator = LineGenerator(lsd_diagonal_size=850),
        ),
        name="PnP+L-LG"
    )

    points_lines_loma = GradableLocalizer(
        creator=PnPLLocalizer.get_creation_function(
            cam2_intrinsic_mtx = headset_recording.intrinsic_cam_mtx,
            extract_and_match_wrapper_config = ExtractAndMatchWrapperConfig(extract_and_match=ExtractAndMatchLoMa('LoMaB128'), crop_augmentations=[0.4]),
            cam1_line_generator = LineGenerator(lsd_diagonal_size=850),
        ),
        name="PnP+L-LoMa"
    )

    ellipsoids_light_glue = GradableLocalizer(
        creator=EllipsoidLocalizer.get_creation_function(
            cam2_intrinsic_mtx = headset_recording.intrinsic_cam_mtx,
            matching_config=GaussianMatchingConfig(dummy_value=0.001),
            cam1_segmenter = SAM3Segmenter(Sam3Prompt(mask_threshold=0.2)),
            cam2_segmenter = YOLOv26Segmenter(),
            ellipsoid_fitter = MVEEEllipsoidFitter(contamination=0.05),
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(extract_and_match=ExtractAndLightGlue(), crop_augmentations=[0.4]),
            ellipsoid_matching_config=PointCloudMatchingConfig(min_cluster_size=2),
        ),
        name="Ellipsoids"
    )

    grader = NPredictors1DatasetGrader(
        gradable_pose_predictors=[points_light_glue, points_loma, ellipsoids_light_glue, points_lines_light_glue, points_lines_loma],
        headset_data = headset_recording,
        robot_env = scanned_3d_env,
        compute_ray_intersection_error=True, use_tqdm_for_frames=True, use_tqdm_for_predictors=False
    )

    print(f"\n\n\n DATASET: {name} \n\n\n")
    grader.print_summary()
    grader.print_error_under_limits(limits_m=[0.05, 0.1, 0.2, 0.5], error_type='ATE')
    grader.print_error_under_limits(limits_m=[2, 5, 10, 15], error_type='ARE', avg_error_fmt = ".1f")

    fig1, axes = plt.subplots(2, 3, figsize = (15, 8))

    grader.plot_signed_error_comparison(axes = [axes[0,0], axes[0,1], axes[0,2], axes[1,0], axes[1,1], axes[1,2]],explain = True)